# 프로토콜 v1.2 — WS2: 반합성 데이터 실험 (탐색 분석, CPU)

**시험 데이터를 쓰지 않습니다.** 개발 데이터로 v1.0 V0를 다시 적합해 함대 값(경로 사전분포, 관측 잡음, 임계값)을 얻고, 그것을 "참값 모집단"으로 삼아 엔진을 만들어 냅니다(v1.2 3절).

- **시나리오:** G0 모형이 맞음 / G1 꺾인 직선 경로(고장 시점은 G0와 같은 분포) / G2 관측 잡음 2배 / G3 학습 엔진을 100사이클에서 중도절단
- **설계:** 학습 엔진 수 m ∈ {2, 3, 5, 10, 20}, 반복 200회(반복마다 학습 m대와 평가 100대를 새로 뽑음), 평가 시점은 수명의 25·50·75%, 서브셋 9개
- **방법:** PI-BDP 재가중 4가지(noWeibull, V0 = 곱, melding α = 0.5, α = 1), 기준(oracle: 참 사전분포로 같은 계산), GB(합성 HI 특징, v1.0 GB의 고른 설정)
- **지표:** RMSE, 90% 포함률·폭(보정 전), interval score, CRPS, 기준 사후분포와의 1-Wasserstein 거리
- **m이 작을 때:** v1.2 2.3절 규칙(m ≤ 3이면 대각 공분산, Weibull이 실패하면 재가중 없음)을 WS2에도 똑같이 씁니다. 쓰인 경우는 모두 기록합니다.
- **계산량 규칙 (v1.2 3.3절):** 첫 서브셋·G0의 10회로 전체 시간을 추정해 24시간을 넘으면 반복을 100회로 줄입니다. 시간만 봅니다.

PI-BDP 라이브러리는 v1.0 고정본(`freeze_v1/pibdp_ncm.py`)을 해시로 확인해 씁니다. 시뮬레이션 코드(`ws2sim.py`)는 이 노트북 안에 들어 있습니다.

**실행 전:** 두 커밋 주소를 넣으세요. **런타임:** CPU, 고용량 RAM(코어가 많을수록 빠름). 끊기면 다시 모두 실행하면 끝난 조합은 건너뜁니다.

In [ ]:
import os, sys, json, glob, time, hashlib, re, shutil
import numpy as np, pandas as pd
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/PI-BDP_NCMAPSS'; LIB = '/content/ws2lib'
else:
    ROOT = os.environ.get('NCM_ROOT', './ncm'); LIB = os.environ.get('WS2_LIB', './ws2lib')
OUT = f'{ROOT}/addendum_v12/ws2'; os.makedirs(f'{OUT}/raw', exist_ok=True); os.makedirs(LIB, exist_ok=True)
URL = r'https://github\.com/.+/commit/[0-9a-f]{7,40}$'
ADDENDUM_V12_URL = 'https://github.com/goniworld/pibdp-ncmapss/commit/f01eecf82f5bc45ed9a90d44d8361b8dcb217d63'
RUN_COMMIT_URL = ''     # 이 노트북을 커밋한 주소
assert re.match(URL, ADDENDUM_V12_URL) and re.match(URL, RUN_COMMIT_URL), '커밋 주소를 넣으세요.'
SUB9 = ['DS01-005', 'DS02-006', 'DS03-012', 'DS04', 'DS05', 'DS06', 'DS07', 'DS08a-009', 'DS08c-008']
SUBSETS = os.environ.get('WS2_SUBSETS', ','.join(SUB9)).split(',')          # 로컬 시험용 환경 변수 (Colab에서는 9개)
SCEN = ['G0', 'G1', 'G2', 'G3']; M_LIST = [2, 3, 5, 10, 20]
N_REP_PLAN, N_EVAL, FRACS, N_SAMP = 200, int(os.environ.get('WS2_EVAL', 100)), [0.25, 0.5, 0.75], 2000
BUDGET_H = 24.0
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 22), b''): h.update(b)
    return h.hexdigest()
print('CPU 코어', os.cpu_count())

In [ ]:
# === 라이브러리: v1.0 고정본 + WS2 시뮬레이션 코드 ===
SIM_SRC = '"""WS2 semi-synthetic experiment (protocol addendum v1.2, section 3). Uses pibdp_ncm (PI-BDP library of v1.0)."""\nimport numpy as np\nfrom scipy.stats import gaussian_kde\nfrom sklearn.ensemble import HistGradientBoostingRegressor\nimport pibdp_ncm as P\n\nQL = np.array([0.025, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.975])\nUG = (np.arange(200) + 0.5) / 200                     # quantile grid for the 1-Wasserstein distance\nMETHODS = [\'noWeibull\', \'V0\', \'meld0.5\', \'meld1\', \'oracle\']\nGRID = np.arange(0, 400.5, 0.5)                       # failure-time grid for the induced prior pi0(T)\n\n\ndef truth_from_fit(m):\n    d = m.deg\n    return dict(mu=d.mu.tolist(), S=d.S.tolist(), sig=float(d.sig), D_mu=float(d.D_mu), D_sd=float(d.D_sd),\n                wb_k=float(m.wb.k), wb_lam=float(m.wb.lam))\n\n\ndef fail_time(th, D):\n    a, b, c = th[..., 0], np.exp(th[..., 1]), np.exp(th[..., 2])\n    arg = (D - a) / b\n    return np.where(arg > 1, np.log(np.maximum(arg, 1)) / c, np.nan) * P.TS\n\n\ndef draw_engines(tr, n, rng, scen):\n    """n engines from the true population; G1: broken-stick path with the same failure times; G2: noise x2."""\n    mu, S = np.array(tr[\'mu\']), np.array(tr[\'S\']); sig = tr[\'sig\'] * (2 if scen == \'G2\' else 1)\n    out = []\n    while len(out) < n:\n        th = rng.multivariate_normal(mu, S); D = rng.normal(tr[\'D_mu\'], tr[\'D_sd\'])\n        T = fail_time(th, D)\n        if not np.isfinite(T) or T < 10 or T > 200: continue\n        eol = int(np.floor(T)); t = np.arange(1, eol + 1, dtype=float)\n        if scen == \'G1\':\n            y0 = P.model(th, 0.0); t0 = rng.uniform(0.3, 0.7) * T\n            path = np.where(t < t0, y0, y0 + (D - y0) * (t - t0) / (T - t0))\n        else:\n            path = P.model(th, t / P.TS)\n        out.append(dict(t=t, y=path + rng.normal(0, sig, len(t)), eol=eol, cens=False))\n    return out\n\n\ndef censor(engs, at=100):\n    out = []\n    for e in engs:\n        if e[\'eol\'] > at:\n            k = int(at); out.append(dict(t=e[\'t\'][:k], y=e[\'y\'][:k], eol=k, cens=True))\n        else:\n            out.append(e)\n    return out\n\n\ndef fit_fleet(train, shrink):\n    """PI-BDP fleet prior, noise, threshold and Weibull from m training engines (v1.2 2.3 small-m rules)."""\n    trajs = [(e[\'t\'], e[\'y\']) for e in train]; failed = [not e[\'cens\'] for e in train]\n    if sum(failed) == 0: return None, None, \'no failed engine\'\n    deg = P.BayesExp(shrink); deg.fit_units(trajs, failed)\n    idx = np.arange(len(train)); note = []\n    if len(train) <= 3:                                # covariance singular -> diagonal (v1.2 2.3)\n        Pm = deg.P_u; deg.mu = np.median(Pm, 0)\n        deg.S = np.diag(np.var(Pm, 0, ddof=1)) + 1e-6 * np.eye(3)\n        deg.Linv = np.linalg.cholesky(np.linalg.inv(deg.S)).T\n        deg.sig = np.std(np.concatenate(deg.R_u))\n        Dv = deg.D_u[~np.isnan(deg.D_u)]; deg.D_mu, deg.D_sd = np.mean(Dv), np.std(Dv) + 1e-9\n        note.append(\'diagonal covariance\')\n    else:\n        deg.aggregate(idx, inplace=True)\n    try:\n        wb = P.WeibullCens().fit([e[\'eol\'] for e in train], [e[\'cens\'] for e in train])\n        if not (np.isfinite(wb.k) and np.isfinite(wb.lam)): raise ValueError\n    except Exception:\n        wb = None; note.append(\'weibull failed -> noWeibull\')\n    return deg, wb, \';\'.join(note)\n\n\ndef oracle_deg(tr, scen):\n    d = P.BayesExp(0.0)\n    d.mu, d.S = np.array(tr[\'mu\']), np.array(tr[\'S\'])\n    d.Linv = np.linalg.cholesky(np.linalg.inv(d.S)).T\n    d.sig = tr[\'sig\'] * (2 if scen == \'G2\' else 1); d.D_mu, d.D_sd = tr[\'D_mu\'], tr[\'D_sd\']\n    return d\n\n\ndef log_pi0(deg, rng, n=20000, kde_n=5000):\n    th = rng.multivariate_normal(deg.mu, deg.S, n, method=\'svd\'); D = rng.normal(deg.D_mu, deg.D_sd, n)\n    T = fail_time(th, D); T = T[np.isfinite(T) & (T > 0)]\n    if len(T) < 50: return None\n    k = gaussian_kde(T[:kde_n])\n    return np.log(np.maximum(k(GRID), 1e-300))\n\n\ndef wq(x, w, q): return P.wquantile(x, w, q)\n\n\ndef scores(x, w, y, ref_q):\n    q = wq(x, w, QL); qg = wq(x, w, UG)\n    L, U = q[1], q[7]\n    return dict(err=q[4] - y, cover=float(L <= y <= U), width=U - L,\n                iscore=(U - L) + 20 * (L - y) * (y < L) + 20 * (y - U) * (y > U),\n                crps=P.wcrps(x, w, y), w1=float(np.mean(np.abs(qg - ref_q))) if ref_q is not None else np.nan)\n\n\ndef run_rep(tr, scen, m, rep_seed, n_eval, fracs, n_samp, shrink, gb_params):\n    rng = np.random.default_rng(rep_seed)\n    train = draw_engines(tr, m, rng, scen)\n    if scen == \'G3\': train = censor(train)\n    evals = draw_engines(tr, n_eval, rng, scen)\n    deg, wb, note = fit_fleet(train, shrink)\n    rows = []\n    if deg is None:\n        return rows, dict(note=note, n_cens=sum(e[\'cens\'] for e in train))\n    lp0 = log_pi0(deg, rng)\n    odeg = oracle_deg(tr, scen)\n    # GB on synthetic HI features (failed training engines only)\n    Xtr, ytr = [], []\n    for e in train:\n        if e[\'cens\']: continue\n        Xtr.append(P.window_features(e[\'y\'][:, None], e[\'t\'])); ytr.append(e[\'eol\'] - e[\'t\'])\n    gb = HistGradientBoostingRegressor(random_state=int(rep_seed % (2 ** 31)), early_stopping=False, **gb_params).fit(np.vstack(Xtr), np.concatenate(ytr))\n    for ei, e in enumerate(evals):\n        F = P.window_features(e[\'y\'][:, None], e[\'t\']); gbp = gb.predict(F)\n        for f in fracs:\n            j = max(1, int(np.ceil(f * e[\'eol\']))); t, y = e[\'t\'][:j], e[\'y\'][:j]; true = e[\'eol\'] - t[-1]\n            ro = odeg.rul_samples(t, y, n_samp, rng); wo = np.ones_like(ro) / len(ro)\n            ref = wq(ro, wo, UG)\n            r = deg.rul_samples(t, y, n_samp, rng); Tn = t[-1] + r\n            W = {\'noWeibull\': np.ones_like(r)}\n            if wb is not None:\n                lw = wb.logpdf(Tn); W[\'V0\'] = np.exp(lw - lw.max())\n                if lp0 is not None:\n                    lr = lw - np.interp(Tn, GRID, lp0)\n                    for a in (0.5, 1.0):\n                        z = a * lr; W[f\'meld{a:g}\'] = np.exp(z - z.max())\n            for k in [\'V0\', \'meld0.5\', \'meld1\']:\n                W.setdefault(k, W[\'noWeibull\'])        # fallback recorded in note\n            for k in [\'noWeibull\', \'V0\', \'meld0.5\', \'meld1\']:\n                rows.append(dict(method=k, frac=f, **scores(r, W[k] / W[k].sum(), true, ref)))\n            rows.append(dict(method=\'oracle\', frac=f, **scores(ro, wo, true, ref)))\n            gpe = gbp[j - 1]\n            rows.append(dict(method=\'GB\', frac=f, err=gpe - true, cover=np.nan, width=np.nan, iscore=np.nan, crps=np.nan, w1=np.nan))\n    return rows, dict(note=note + (\'\' if lp0 is not None else \';pi0 failed -> meld=noWeibull\'), n_cens=sum(e[\'cens\'] for e in train))\n\n\ndef aggregate(rows):\n    import pandas as pd\n    d = pd.DataFrame(rows)\n    return d.groupby([\'method\', \'frac\']).agg(sse=(\'err\', lambda v: float(np.sum(np.square(v)))), n=(\'err\', \'size\'),\n                                             cover=(\'cover\', \'mean\'), width=(\'width\', \'mean\'), iscore=(\'iscore\', \'mean\'),\n                                             crps=(\'crps\', \'mean\'), w1=(\'w1\', \'mean\')).reset_index()\n'
MAN = json.load(open(f'{ROOT}/freeze_v1/MANIFEST.json'))
assert sha(f'{ROOT}/freeze_v1/pibdp_ncm.py') == MAN['library_sha256'], '라이브러리가 v1.0 고정본과 다릅니다.'
shutil.copy(f'{ROOT}/freeze_v1/pibdp_ncm.py', f'{LIB}/pibdp_ncm.py'); open(f'{LIB}/ws2sim.py', 'w').write(SIM_SRC)
sys.path.insert(0, LIB); os.environ['PYTHONPATH'] = LIB + os.pathsep + os.environ.get('PYTHONPATH', '')
import importlib, pibdp_ncm as P, ws2sim as W; importlib.reload(P); importlib.reload(W)
json.dump(dict(pibdp_ncm=MAN['library_sha256'], ws2sim=hashlib.sha256(SIM_SRC.encode()).hexdigest()), open(f'{OUT}/ws2_code_sha256.json', 'w'), indent=1)
cfg = json.load(open(f'{ROOT}/phase1/config.json')); GB_PARAMS = json.load(open(f'{ROOT}/phase2/ml_selected.json'))['GB']['params']
print('라이브러리 = v1.0 고정본 확인 | 설정', cfg['baseline'], cfg['n_h'], cfg['shrink'], '| GB', GB_PARAMS)

In [ ]:
# === 참값 모집단: 개발 엔진 전체로 V0를 다시 적합 (v1.0 Phase 3와 같은 호출) ===
rec0 = json.load(open(f'{ROOT}/phase0/sha256_phase0.json')); TF = f'{OUT}/ws2_truth.json'
TRUTH = json.load(open(TF)) if os.path.exists(TF) else {}
for s in SUBSETS:
    if s in TRUTH: continue
    f = f'{ROOT}/phase0/dev_{s}.npz'; assert sha(f) == rec0[f'dev_{s}.npz'], f'{s}: 개발 파일 해시 불일치'
    Dd = P.load_dev(f)
    m = P.PIBDP(baseline=cfg['baseline'], n_h=cfg['n_h'], k=cfg['k'][s], shrink=cfg['shrink']).fit(Dd, list(Dd['units']))
    TRUTH[s] = W.truth_from_fit(m)
    eol = Dd['eol'][~Dd['censored']]; TRUTH[s]['dev_eol_mean'] = float(eol.mean()); TRUTH[s]['dev_eol_sd'] = float(eol.std())
    json.dump(TRUTH, open(TF, 'w'), indent=1); print(s, '참값 함대 값 저장')
# 참 모집단이 만드는 수명 분포를 실제 개발 수명과 나란히 (확인용)
chk = []
for s in SUBSETS:
    e = W.draw_engines(TRUTH[s], 2000, np.random.default_rng(1), 'G0'); L = np.array([x['eol'] for x in e])
    chk.append(dict(subset=s, sim_eol_mean=L.mean(), sim_eol_sd=L.std(), dev_eol_mean=TRUTH[s]['dev_eol_mean'], dev_eol_sd=TRUTH[s]['dev_eol_sd'],
                    sim_frac_over_100=float((L > 100).mean())))
CHK = pd.DataFrame(chk); CHK.to_csv(f'{OUT}/ws2_truth_check.csv', index=False); print(CHK.round(2).to_string(index=False))

In [ ]:
# === 한 조합 실행 (서브셋, 시나리오, m) — 반복을 병렬로 ===
from joblib import Parallel, delayed
def seed(s, g, m, r): return int(np.random.SeedSequence([2026, SUB9.index(s), SCEN.index(g), m, r]).generate_state(1)[0])
def one(s, g, m, r):
    rows, info = W.run_rep(TRUTH[s], g, m, seed(s, g, m, r), N_EVAL, FRACS, N_SAMP, cfg['shrink'], GB_PARAMS)
    a = W.aggregate(rows) if rows else pd.DataFrame()
    return a.assign(subset=s, scen=g, m=m, rep=r, note=info['note'], n_cens=info['n_cens']) if len(a) else \
        pd.DataFrame([dict(subset=s, scen=g, m=m, rep=r, note=info['note'], n_cens=info['n_cens'])])
def run_combo(s, g, m, n_rep):
    f = f'{OUT}/raw/{s}_{g}_m{m}.csv.gz'
    if os.path.exists(f): return
    res = Parallel(n_jobs=-1)(delayed(one)(s, g, m, r) for r in range(n_rep))
    pd.concat(res, ignore_index=True).to_csv(f, index=False)

In [ ]:
# === 계산량 규칙: 첫 서브셋·G0, 10회 (m을 돌아가며) — 시간만 봄 ===
DEC = f'{OUT}/ws2_compute_decision.json'
if not os.path.exists(DEC):
    s0 = SUBSETS[0]; t = time.time()
    Parallel(n_jobs=-1)(delayed(one)(s0, 'G0', M_LIST[r % 5], 10_000 + r) for r in range(10))     # 시간 측정용 (결과는 버림)
    per10 = time.time() - t
    est_h = per10 / 10 * len(SUBSETS) * len(SCEN) * len(M_LIST) * N_REP_PLAN / 3600
    n_rep = N_REP_PLAN if est_h <= BUDGET_H else 100
    json.dump(dict(cores=os.cpu_count(), sec_per_10_reps_parallel=per10, est_hours_200=est_h, budget_h=BUDGET_H, n_rep=n_rep,
                   rule='v1.2 3.3: >24 h -> 100 reps (timing only)'), open(DEC, 'w'), indent=1)
dec = json.load(open(DEC)); N_REP = int(os.environ.get('WS2_REPS', dec['n_rep'])); print(dec, '| 반복', N_REP)

In [ ]:
# === 본 실행 (끊기면 다시 실행: 끝난 조합은 건너뜀) ===
t0 = time.time()
for s in SUBSETS:
    for g in SCEN:
        for m in M_LIST:
            tt = time.time(); run_combo(s, g, m, N_REP)
            print(f'{s} {g} m={m}  {time.time() - tt:.0f}s  (누적 {(time.time() - t0) / 3600:.2f} h)')
print('완료:', len(glob.glob(f'{OUT}/raw/*.csv.gz')), '/', len(SUBSETS) * len(SCEN) * len(M_LIST))

In [ ]:
# === 집계: 시나리오 × m × 방법 (서브셋·반복·평가 시점을 모두 묶음) ===
R = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(f'{OUT}/raw/*.csv.gz'))], ignore_index=True)
skipped = R[R.method.isna()] if 'method' in R else R.iloc[0:0]
R = R[R.method.notna()]
def summ(g):
    return pd.Series(dict(rmse=np.sqrt(g.sse.sum() / g.n.sum()), cov90=g.cover.mean(), width90=g.width.mean(), iscore90=g.iscore.mean(),
                          crps=g.crps.mean(), w1=g.w1.mean(), n_rep=g.rep.nunique() * 0 + len(g[['subset', 'rep']].drop_duplicates())))
TAB = R.groupby(['scen', 'm', 'method']).apply(summ).reset_index()
# 반복 단위 표준오차 (서브셋·반복별 값을 단위로)
U = R.groupby(['scen', 'm', 'method', 'subset', 'rep']).apply(lambda g: pd.Series(dict(cov=g.cover.mean(), isc=g.iscore.mean(), w1=g.w1.mean()))).reset_index()
SE = U.groupby(['scen', 'm', 'method']).agg(cov_se=('cov', lambda v: v.std(ddof=1) / np.sqrt(len(v))), isc_se=('isc', lambda v: v.std(ddof=1) / np.sqrt(len(v))),
                                            w1_se=('w1', lambda v: v.std(ddof=1) / np.sqrt(len(v)))).reset_index()
TAB = TAB.merge(SE, on=['scen', 'm', 'method'], how='left'); TAB.to_csv(f'{OUT}/ws2_table.csv', index=False)
BYF = R.groupby(['scen', 'm', 'method', 'frac']).apply(summ).reset_index(); BYF.to_csv(f'{OUT}/ws2_by_frac.csv', index=False)
BYS = R.groupby(['scen', 'subset', 'm', 'method']).apply(summ).reset_index(); BYS.to_csv(f'{OUT}/ws2_by_subset.csv', index=False)
NOTES = R.drop_duplicates(['subset', 'scen', 'm', 'rep']).groupby(['scen', 'm']).agg(
    diag_cov=('note', lambda v: v.fillna('').str.contains('diagonal').mean()), wb_fail=('note', lambda v: v.fillna('').str.contains('weibull').mean()),
    pi0_fail=('note', lambda v: v.fillna('').str.contains('pi0').mean()), mean_censored=('n_cens', 'mean')).reset_index()
NOTES.to_csv(f'{OUT}/ws2_fallbacks.csv', index=False)
for k in ['rmse', 'cov90', 'iscore90', 'w1']:
    print(f'\n[{k}]'); print(TAB.pivot_table(index=['scen', 'm'], columns='method', values=k).round(3).to_string())
print('\n건너뛴 반복 (실패한 고장 엔진 없음 등):', len(skipped)); print(NOTES.round(3).to_string(index=False))

In [ ]:
# === WS4의 예측 확인 (v1.2 3.6) — 기술 통계 ===
P4 = TAB.pivot_table(index=['scen', 'm'], columns='method', values=['cov90', 'width90', 'w1'])
Q = pd.DataFrame({'V0_cov - meld1_cov': P4['cov90']['V0'] - P4['cov90']['meld1'],
                  'V0_width - meld1_width': P4['width90']['V0'] - P4['width90']['meld1'],
                  'V0_w1 - meld1_w1': P4['w1']['V0'] - P4['w1']['meld1'],
                  'V0_w1 - noW_w1': P4['w1']['V0'] - P4['w1']['noWeibull']}).reset_index()
Q.to_csv(f'{OUT}/ws2_ws4_check.csv', index=False); print(Q.round(3).to_string(index=False))
txt = '\n'.join([f"Addendum v1.2 WS2 (exploratory, synthetic) {time.strftime('%Y-%m-%d %H:%M:%S')}  addendum={ADDENDUM_V12_URL}  run={RUN_COMMIT_URL}",
    f"Compute decision: {json.dumps(dec)}; reps used {N_REP}; eval engines {N_EVAL}; fractions {FRACS}; samples {N_SAMP}",
    '', 'Truth check (simulated vs dev lifetimes):', CHK.round(2).to_string(index=False),
    *sum([['', f'[{k}] by scenario x m x method:', TAB.pivot_table(index=['scen', 'm'], columns='method', values=k).round(3).to_string()]
          for k in ['rmse', 'cov90', 'width90', 'iscore90', 'crps', 'w1']], []),
    '', 'WS4 predictions (V0 vs melding):', Q.round(3).to_string(index=False),
    '', 'Fallback rates:', NOTES.round(3).to_string(index=False)])
open(f'{OUT}/ws2_summary.txt', 'w').write(txt)
H = {os.path.relpath(p, OUT): sha(p) for p in sorted(glob.glob(f'{OUT}/**/*.*', recursive=True)) if not p.endswith('sha256_ws2.json')}
json.dump(H, open(f'{OUT}/sha256_ws2.json', 'w'), indent=1); print(txt)